# Model and Export

## Setup & Imports

In [1]:

# All imports in one place
import json, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor, Pool
from pathlib import Path
import joblib
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import FunctionTransformer
import ssl
import certifi

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def print_metrics(y_true, y_pred, title="Metrics"):
    mae = mean_absolute_error(y_true, y_pred)
    med = median_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    print(f"{title}:\n  MAE={mae:.2f} | MedAE={med:.2f} | R2={r2:.4f}")

print("Imports OK.")


Imports OK.


## Load Data

In [2]:
DATA_PATH = Path("../data/raw")
raw_path = DATA_PATH / "rents.csv" 

df = pd.read_csv(raw_path)

for c in ['address','district','type']:
    df[c] = df[c].astype('string')
df['total'] = pd.to_numeric(df['total'], errors='coerce')
df.dropna(subset=['total'], inplace=True)


## Geocoding

In [3]:
# --- Definição dos Caminhos ---
# Define o diretório de onde os dados brutos são lidos
raw_data_dir = Path('../data/raw')
# Define o diretório de onde os dados processados serão salvos/lidos
work_data_dir = Path('../data/work')

# Cria o diretório de trabalho se ele não existir
work_data_dir.mkdir(parents=True, exist_ok=True)

# Define o caminho completo para os arquivos
original_file = raw_data_dir / 'rents.csv'
geocoded_file = work_data_dir / 'rents_geocoded.csv'
# ---------------------------------

# Lógica para carregar ou criar o arquivo geocodificado
if geocoded_file.exists():
    # Se o arquivo JÁ EXISTE, simplesmente o carregamos (rápido)
    print(f"Arquivo geocodificado '{geocoded_file}' encontrado. Carregando...")
    df = pd.read_csv(geocoded_file)
    print("Carregamento concluído.")
else:
    # Se o arquivo NÃO EXISTE, executamos o processo demorado
    print(f"Arquivo geocodificado não encontrado. Iniciando processo a partir de '{original_file}'...")
    df = pd.read_csv(original_file)

    # --- Início do Bloco de Geocodificação ---
    # (Este é o código que você quer que rode apenas uma vez)
    ctx = ssl.create_default_context(cafile=certifi.where())
    geolocator = Nominatim(user_agent="aluguel_sp_app_v5", ssl_context=ctx)
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    
    df['full_address'] = df['address'] + ', ' + df['district'] + ', São Paulo, Brazil'
    unique_addresses = df['full_address'].unique()
    location_dict = {}

    for address in tqdm(unique_addresses):
        try:
            location = geocode(address, timeout=20)
            if location:
                location_dict[address] = (location.latitude, location.longitude)
            else:
                location_dict[address] = (None, None)
        except Exception as e:
            location_dict[address] = (None, None)

    df['latitude']  = df['full_address'].map(location_dict).str[0]
    df['longitude'] = df['full_address'].map(location_dict).str[1]
    df['latitude'] = df.groupby('district')['latitude'].transform(lambda x: x.fillna(x.median()))
    df['longitude'] = df.groupby('district')['longitude'].transform(lambda x: x.fillna(x.median()))
    df.dropna(subset=['latitude', 'longitude'], inplace=True)
    
    # Ao final, salvamos o arquivo para as próximas execuções
    df.to_csv(geocoded_file, index=False)
    print(f"\nGeocodificação finalizada. Arquivo '{geocoded_file}' salvo para uso futuro.")
    # --- Fim do Bloco de Geocodificação ---

# Ao final desta célula, o DataFrame 'df' estará pronto para ser usado.
df.head()

Arquivo geocodificado '..\data\work\rents_geocoded.csv' encontrado. Carregando...
Carregamento concluído.


,address,district,area,bedrooms,garage,type,rent,total,full_address,latitude,longitude,geo_cluster
0,Avenida São Miguel,Vila Marieta,15,1,1,Studio e kitnet,1030,1345,"Avenida São Miguel, Vila Marieta, São Paulo, B...",-23.515052,-46.520220,143
1,Rua Oscar Freire,Pinheiros,18,1,0,Apartamento,4000,4661,"Rua Oscar Freire, Pinheiros, São Paulo, Brazil",-23.550754,-46.678799,41
2,Rua Júlio Sayago,Vila Ré,56,2,2,Casa em condomínio,1750,1954,"Rua Júlio Sayago, Vila Ré, São Paulo, Brazil",-23.525495,-46.502078,92
3,Rua Barata Ribeiro,Bela Vista,19,1,0,Studio e kitnet,4000,4654,"Rua Barata Ribeiro, Bela Vista, São Paulo, Brazil",-23.556522,-46.653316,96
4,Rua Domingos Paiva,Brás,50,2,1,Apartamento,3800,4587,"Rua Domingos Paiva, Brás, São Paulo, Brazil",-23.544782,-46.617143,99


## Saving the Dataframe with the geocoding

In [4]:
if not geocoded_file.exists():
    df.to_csv(geocoded_file, index=False)
    print(f"\nSaving File '{geocoded_file}'.")

## Geo-Clustering

In [5]:
coords = df[['latitude', 'longitude']]

kmeans = KMeans(n_clusters=150, random_state=42, n_init=10)
df['geo_cluster'] = kmeans.fit_predict(coords)

  File "c:\DEV\brazil-rent-price-estimator\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\henrique.nascimento\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 546, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\henrique.nascimento\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1022, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\henrique.nascimento\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1491, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


## Train/Test split (stratified by `type`)

In [6]:
features = [
    'address','district','type','area','bedrooms','garage', # Features originais
    'latitude', 'longitude', 'geo_cluster' # NOVAS FEATURES
]
target   = 'total'

X = df[features].copy()
y = df[target].astype(float).copy()

strat = X['type'].fillna('NA').astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=strat
)

print(f"Dimensões Originais - Treino: {X_train.shape}, Teste: {X_test.shape}")

Dimensões Originais - Treino: (6728, 9), Teste: (1682, 9)


## Cleaning Data

In [7]:
df_train_temp = X_train.copy()
df_train_temp['total'] = y_train

original_train_count = len(df_train_temp)

print(f"\nOriginal traingin shape: {original_train_count} samples")

area_min = 10
area_max = 1500
df_train_temp = df_train_temp[(df_train_temp['area'] >= area_min) & (df_train_temp['area'] <= area_max)]

preco_m2 = df_train_temp['total'] / df_train_temp['area'].clip(lower=1)
m2_min = 20
m2_max = 400
df_train_temp = df_train_temp[(preco_m2 >= m2_min) & (preco_m2 <= m2_max)]

X_train = df_train_temp[features]
y_train = df_train_temp[target]

print(f"Training shape after cleaning {len(X_train)} samples")

print(f"{original_train_count - len(X_train)} samples removed after cleaning")


Original traingin shape: 6728 samples
Training shape after cleaning 6411 samples
317 samples removed after cleaning


## Preprocessing blocks

In [8]:

num_cols = ['area','bedrooms','garage']
cat_cols = ['address','district','type']

preprocess_ohe = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False), cat_cols),
    ],
    remainder='drop',
    verbose_feature_names_out=False
)


## Linear Regression (baseline)

In [9]:

lin_pipe = Pipeline([('preprocess', preprocess_ohe),
                     ('model', LinearRegression())])
lin_pipe.fit(X_train, y_train)
y_pred_lin = lin_pipe.predict(X_test)
print_metrics(y_test, y_pred_lin, title='Linear Regression (baseline)')


Linear Regression (baseline):
  MAE=1595149117144.39 | MedAE=1328.36 | R2=-688162765128487040.0000


## Random Forest Regressor

In [10]:

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_pipe = Pipeline([('preprocess', preprocess_ohe),
                    ('model', rf)])
rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)
print_metrics(y_test, y_pred_rf, title='Random Forest')


Random Forest:
  MAE=1169.78 | MedAE=693.26 | R2=0.7112


## Linear (Tuned) — Ridge with hyperparameter search

In [11]:

# Build a slightly different preprocessor with scaling on numerics
preprocess_ohe_scale = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False), cat_cols),
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

ridge_pipe = Pipeline([('preprocess', preprocess_ohe_scale),
                       ('model', Ridge(random_state=RANDOM_STATE))])

param_grid = {
    'model__alpha': [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
}

gs = GridSearchCV(ridge_pipe, param_grid=param_grid, scoring='neg_mean_absolute_error', cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

ridge_best = gs.best_estimator_
y_pred_ridge = ridge_best.predict(X_test)
print("Best alpha:", gs.best_params_)
print_metrics(y_test, y_pred_ridge, title='Ridge (tuned)')


Best alpha: {'model__alpha': 0.1}
Ridge (tuned):
  MAE=1272.50 | MedAE=858.10 | R2=0.6859


## CatBoost Pipeline with Geo-Features

In [12]:


# --- Pipeline para o CatBoost com Geo-Features ---

# Função para criar features como 'area_log', etc.
def create_extra_features(df):
    df_out = df.copy()
    df_out['area_log'] = np.log1p(df_out['area'])
    # Usar .clip para evitar divisão por zero se houver 0 quartos
    df_out['area_per_bedroom'] = df_out['area'] / np.clip(df_out['bedrooms'].astype(float), 1, None)
    df_out['district_type'] = df_out['district'].astype(str) + '_' + df_out['type'].astype(str)
    return df_out

# Listas de colunas para o modelo final
feat_cols_cb = [
    'area','bedrooms','garage','area_log','area_per_bedroom',
    'type','district', 'address', 'district_type',
    'latitude', 'longitude', 'geo_cluster'
]
cat_cols_cb  = [
    'type','district', 'address', 'district_type',
    'geo_cluster'
]


# >>>>> INÍCIO DA CORREÇÃO <<<<<
# Precisamos encontrar os ÍNDICES das colunas categóricas na lista final de features
cat_features_indices = [feat_cols_cb.index(col) for col in cat_cols_cb]


# O ColumnTransformer garante que apenas as colunas certas sejam passadas para o modelo
preprocessor_cb = ColumnTransformer(
    transformers=[('select_features', 'passthrough', feat_cols_cb)],
    remainder='drop'
)

# Criando o pipeline final do CatBoost
pipe_catboost_geo = Pipeline(steps=[
    ('add_features', FunctionTransformer(create_extra_features)),
    ('select_cols', preprocessor_cb),
    ('model', CatBoostRegressor(
        # Passamos os índices das colunas categóricas AQUI
        cat_features=cat_features_indices,
        loss_function='MAE', eval_metric='MAE',
        learning_rate=0.04, depth=8, l2_leaf_reg=3.0,
        iterations=1500,
        random_seed=RANDOM_STATE, verbose=0,
        od_type='Iter', od_wait=300
    ))
])
# >>>>> FIM DA CORREÇÃO <<<<<

print("Pipeline 'pipe_catboost_geo' criado com sucesso e pronto para o Stacking.")

Pipeline 'pipe_catboost_geo' criado com sucesso e pronto para o Stacking.


## Training and Prediction

In [13]:
# Usamos os dados de treino JÁ LIMPOS
X_train_cb = X_train.copy()
X_test_cb = X_test.copy() 

# Engenharia de features (area_log, etc.)
X_train_cb['area_log'] = np.log1p(X_train_cb['area'])
X_test_cb['area_log']  = np.log1p(X_test_cb['area'])
X_train_cb['area_per_bedroom'] = X_train_cb['area'] / np.clip(X_train_cb['bedrooms'].astype(float), 1, None)
X_test_cb['area_per_bedroom']  = X_test_cb['area']  / np.clip(X_test_cb['bedrooms'].astype(float), 1, None)
X_train_cb['district_type'] = X_train_cb['district'].astype(str) + '_' + X_train_cb['type'].astype(str)
X_test_cb['district_type']  = X_test_cb['district'].astype(str) + '_' + X_test_cb['type'].astype(str)

# Definição das colunas para o modelo (incluindo as geo-features se você as tiver)
feat_cols_cb = [
    'area','bedrooms','garage','area_log','area_per_bedroom','type','district', 'address', 'district_type'
    # Adicione 'latitude', 'longitude', 'geo_cluster' se você as criou
]
cat_cols_cb  = [
    'type','district', 'address', 'district_type'
    # Adicione 'geo_cluster' se você a criou
]

# Divisão treino/validação para o early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_cb, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=X_train_cb['type'].astype(str)
)

# Transformação do Alvo (log1p)
y_tr_m2_log  = np.log1p(y_tr / np.clip(X_tr['area'].astype(float), 1, None))
y_val_m2_log = np.log1p(y_val / np.clip(X_val['area'].astype(float), 1, None))

# >>> É AQUI QUE AS VARIÁVEIS SÃO CRIADAS <<<
train_pool = Pool(X_tr[feat_cols_cb], label=y_tr_m2_log, cat_features=cat_cols_cb)
val_pool   = Pool(X_val[feat_cols_cb], label=y_val_m2_log, cat_features=cat_cols_cb)
test_pool  = Pool(X_test_cb[feat_cols_cb], cat_features=cat_cols_cb)

print("Pools de dados do CatBoost criados com sucesso.")


cbr = CatBoostRegressor(
    loss_function='MAE', eval_metric='MAE',
    learning_rate=0.04, 
    depth=8,
    l2_leaf_reg=3.0,
    iterations=5000,
    random_seed=RANDOM_STATE,
    verbose=200,
    od_type='Iter',
    od_wait=300 
)

print("Iniciando o treinamento do CatBoost otimizado...")
cbr.fit(train_pool, eval_set=val_pool)


pred_m2_log_test = cbr.predict(test_pool)

pred_m2_test = np.expm1(pred_m2_log_test)
y_pred_cb = np.clip(pred_m2_test, 0, None) * X_test_cb['area'].astype(float).values

print("\nMétricas do modelo otimizado:")
print_metrics(y_test, y_pred_cb, title='CatBoost Otimizado')

Pools de dados do CatBoost criados com sucesso.
Iniciando o treinamento do CatBoost otimizado...
0:	learn: 0.3903096	test: 0.3989810	best: 0.3989810 (0)	total: 177ms	remaining: 14m 46s
200:	learn: 0.2029338	test: 0.2252575	best: 0.2252474 (199)	total: 5.56s	remaining: 2m 12s
400:	learn: 0.1818102	test: 0.2207065	best: 0.2207065 (400)	total: 11s	remaining: 2m 5s
600:	learn: 0.1685302	test: 0.2195566	best: 0.2194790 (592)	total: 16.5s	remaining: 2m
800:	learn: 0.1587458	test: 0.2192069	best: 0.2192057 (797)	total: 21.9s	remaining: 1m 54s
1000:	learn: 0.1511363	test: 0.2193755	best: 0.2190856 (834)	total: 27.1s	remaining: 1m 48s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.2190856188
bestIteration = 834

Shrink model to first 835 iterations.

Métricas do modelo otimizado:
CatBoost Otimizado:
  MAE=1041.50 | MedAE=549.47 | R2=0.7362


In [14]:
y_pred_ensemble = (y_pred_rf + y_pred_cb) / 2
print("\n--- Ensemble Result: ---")
print_metrics(y_test, y_pred_ensemble, title='Ensemble (RF + CatBoost)')


--- Ensemble Result: ---
Ensemble (RF + CatBoost):
  MAE=1063.00 | MedAE=585.21 | R2=0.7439


## Stacking Models

In [ ]:
# --- Stacking: Combinando os Melhores Modelos ---

# 1. Defina os modelos base que farão as previsões iniciais.
#    Usaremos o Random Forest (pipe_rf já existe no seu notebook) e nosso novo pipeline do CatBoost.
estimators = [
    ('random_forest', rf_pipe),
    ('catboost_geo', pipe_catboost_geo)
]

# 2. Defina o meta-modelo (final_estimator)
#    Ele aprenderá a melhor forma de combinar as previsões dos modelos base.
#    Um modelo linear como o Ridge costuma funcionar muito bem aqui.
stack_reg = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5, # Validação cruzada para gerar as previsões dos modelos base
    n_jobs=-1 # Usar todos os processadores disponíveis
)

# 3. Treine o modelo Stacking
#    Isso vai treinar o RF e o CatBoost, e depois treinar o Ridge sobre as previsões deles.
print("Iniciando treinamento do modelo Stacking...")
stack_reg.fit(X_train, y_train)
print("Treinamento finalizado.")

# 4. Avalie a performance no conjunto de teste
y_pred_stack = stack_reg.predict(X_test)
print_metrics(y_test, y_pred_stack, title='Stacking Regressor (RF + CatBoost com Geo)')

Iniciando treinamento do modelo Stacking...


## Side-by-side comparison

In [ ]:

results = []
results.append(('Linear', mean_absolute_error(y_test, y_pred_lin)))
results.append(('RandomForest', mean_absolute_error(y_test, y_pred_rf)))
results.append(('Ridge (tuned)', mean_absolute_error(y_test, y_pred_ridge)))
results.append(('CatBoost m²', mean_absolute_error(y_test, y_pred_cb)))
pd.DataFrame(results, columns=['model','MAE']).sort_values('MAE')


,model,MAE
3,CatBoost m²,1.041501e+03
1,RandomForest,1.169780e+03
2,Ridge (tuned),1.272503e+03
0,Linear,1.595149e+12


## Error Analysis

In [ ]:
df_analise = X_test.copy()
df_analise['real_total'] = y_test
df_analise['predicted_error'] = y_pred_cb
df_analise['error_abs'] = abs(df_analise['real_total'] - df_analise['predicted_error'])

worst_errors = df_analise.sort_values(by='error_abs', ascending=False)

# Exiba os 20 piores erros
print("Analysing the 20 worst errors:")
worst_errors.head(20)

Analysing the 20 worst errors:


,address,district,type,area,bedrooms,garage,latitude,longitude,geo_cluster,real_total,predicted_error,error_abs
4787,Rua Haddock Lobo,Cerqueira César,Apartamento,41,1,1,-23.562038,-46.665990,79,17930.0,5147.549267,12782.450733
8253,Rua Cajaíba,Vila Pompéia,Apartamento,540,4,4,-23.535421,-46.690765,100,13820.0,25227.583438,11407.583438
7147,Rua Visconde de Porto Seguro,Jardim dos Estados,Casa em condomínio,1,4,3,-23.639966,-46.679668,22,11420.0,59.696823,11360.303177
7343,Rua Barão do Triunfo,Brooklin,Apartamento,78,1,1,-23.626192,-46.685152,102,16560.0,6529.453434,10030.546566
8032,Augusta,Bela Vista,Apartamento,100,2,2,-23.552816,-46.653914,96,16390.0,6453.264158,9936.735842
6795,Avenida Giovanni Gronchi,Vila Andrade,Apartamento,287,3,4,-23.633178,-46.737069,85,20240.0,10512.412552,9727.587448
4773,Rua Abílio Soares,Paraíso,Apartamento,155,2,1,-23.573936,-46.644035,0,17900.0,8173.312268,9726.687732
7542,Rua Senador César Lacerda Vergueiro,Sumarezinho,Studio e kitnet,70,1,1,-23.549949,-46.691902,149,16680.0,7016.854022,9663.145978
7759,Rua Percílio Neto,Vila Gumercindo,Apartamento,111,3,2,-23.609670,-46.619615,44,16290.0,6764.254731,9525.745269
8381,Rua Verbo Divino,Chácara Santo Antônio,Apartamento,242,4,4,-23.637393,-46.698619,102,19580.0,10206.855786,9373.144214


## Export best model

In [ ]:
models_dir = Path("../models")
model_filename = "model.joblib"
model_path = models_dir / model_filename

models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(stack_reg, model_path)

print(f"Best model saved! {model_path}")


PicklingError: Can't pickle <function create_extra_features at 0x00000231E5BCA340>: it's not the same object as __main__.create_extra_features